# Notebook 2: Customer-Support Escalation Workflow

A different topology from the SDLC pipeline: a linear escalation chain
with a human fallback branch.

```
triage → auto-responder → quality-check → (resolved | escalation → human)
```

This notebook demonstrates:
1. A chain topology with a conditional escalation branch
2. How depth costs autonomy — the chain gets fragile fast
3. The masking problem in customer-facing pipelines
4. When the agent should stop and escalate (optimal stopping)
5. What happens when drift hits a production pipeline

In [ ]:
from minimal_oversight import analyze_pipeline
from minimal_oversight.models import Node, PipelineGraph, AggregationType
from minimal_oversight.capacity import check_feasibility, compute_pipeline_capacity
from minimal_oversight.topology import detect_motifs, rank_nodes_by_risk
from minimal_oversight.intervention import (
    explain_failure_surface, diagnose_failure_mode, FailureMode
)
from minimal_oversight.simulation import simulate_pipeline, SimulationConfig
from minimal_oversight import _formulae as F

import numpy as np

## 1. Define the pipeline

A customer-support pipeline where:
- **Triage** classifies the incoming ticket (intent detection)
- **Auto-responder** drafts and sends a response
- **Quality check** reviews the response before delivery
- **Escalation** routes failures to a human agent

Each node has different competence — triage is strong, auto-response is weaker,
and the quality checker acts as corrector with moderate catch rate.

In [ ]:
def make_support_pipeline(
    triage_skill=0.85,
    responder_skill=0.60,
    qc_skill=0.55,
    escalation_skill=0.50,
    qc_catch=0.70,
    review_cap=0.60,
):
    """Customer-support escalation pipeline."""
    triage = Node(
        "triage", sigma_skill=triage_skill,
        catch_rate=0.50, review_capacity=0.30,
    )
    responder = Node(
        "auto_responder", sigma_skill=responder_skill,
        catch_rate=qc_catch, review_capacity=review_cap,
    )
    qc = Node(
        "quality_check", sigma_skill=qc_skill,
        catch_rate=qc_catch, review_capacity=review_cap,
    )
    escalation = Node(
        "escalation", sigma_skill=escalation_skill,
        catch_rate=0.90,  # human catch rate is high
        review_capacity=1.0,  # humans review everything that reaches them
    )

    p = PipelineGraph([triage, responder, qc, escalation])
    p.add_edge("triage", "auto_responder")
    p.add_edge("auto_responder", "quality_check")
    p.add_edge("quality_check", "escalation")
    return p

pipeline = make_support_pipeline()
print(pipeline)
print(f"Depth: {pipeline.depth}")
print(f"Topology: linear chain (escalation)")

## 2. Full analysis

In [ ]:
report = analyze_pipeline(pipeline, p_min=0.75)
print(report)

## 3. Depth costs autonomy

The paper's key insight for chains (Motif 2): masking increases at each layer,
and quality degrades. What happens as we add more processing stages?

In [ ]:
# Build chains of increasing depth with identical nodes
print(f"{'Depth':<8} {'C_op':>8} {'M*_total':>10} {'T*_auto':>10}")
print("-" * 40)

for depth in range(1, 7):
    nodes = [
        Node(f"layer_{i}", sigma_skill=0.60, catch_rate=0.65, review_capacity=0.50)
        for i in range(depth)
    ]
    chain = PipelineGraph(nodes)
    for i in range(depth - 1):
        chain.add_edge(f"layer_{i}", f"layer_{i+1}")

    r = analyze_pipeline(chain, p_min=0.50)
    
    # Total masking: product of per-layer M*
    m_total = 1.0
    for name in chain.topological_order():
        node = chain.get_node(name)
        if node.masking_index and node.masking_index > 0:
            m_total *= node.masking_index
    
    t_auto = r.intervention_schedule[-1].t_auto if r.intervention_schedule else 0
    print(f"{depth:<8} {r.feasibility.c_op:>8.3f} {m_total:>10.2f} {t_auto:>10.1f}")

print("\nKey: masking compounds super-multiplicatively with depth.")
print("The dashboard reads high quality while raw competence silently degrades.")

## 4. Critical depth: when to stop adding layers

D_max tells us: beyond this depth, adding layers hurts quality
even with correctors at every stage.

In [ ]:
print("D_max for different agent/corrector quality:\n")
print(f"{'σ_skill':<10} {'c=0.50':>8} {'c=0.65':>8} {'c=0.80':>8} {'c=0.90':>8}")
print("-" * 45)

for skill in [0.45, 0.55, 0.65, 0.75, 0.85]:
    depths = []
    for catch in [0.50, 0.65, 0.80, 0.90]:
        d = F.max_pipeline_depth(skill, catch, p_min=0.50)
        depths.append(f"{d:>8.1f}")
    print(f"{skill:<10.2f} {''.join(depths)}")

print("\nBetter correctors (higher c) double the useful depth.")
print("Better agents (higher σ) also increase it, but less dramatically.")

## 5. The masking problem in customer support

In a support pipeline, masking is especially dangerous:
the quality-check node catches errors and makes the auto-responder
look better than it is. If the QC node degrades (model drift, 
distribution shift), the apparent quality stays high while
real quality collapses.

In [ ]:
print("What happens when the quality checker's catch rate drops?\n")
print(f"{'QC catch':>10} {'σ_raw(resp)':>12} {'σ_corr(resp)':>13} {'M*(resp)':>10} {'C_op':>8}")
print("-" * 58)

for qc_catch in [0.90, 0.70, 0.50, 0.30, 0.10]:
    p = make_support_pipeline(qc_catch=qc_catch)
    r = analyze_pipeline(p, p_min=0.75)
    
    resp = p.get_node("auto_responder")
    sr = resp.sigma_raw or 0
    sc = resp.sigma_corr or 0
    m = resp.masking_index or 0
    
    print(f"{qc_catch:>10.2f} {sr:>12.3f} {sc:>13.3f} {m:>10.2f} {r.feasibility.c_op:>8.3f}")

print("\nσ_raw stays constant — the agent hasn't changed.")
print("But σ_corr and M* drop as the corrector weakens.")
print("Dual-σ tracking catches this; single-σ (monitoring only σ_corr) misses it.")

## 6. Diagnostic differential: detecting failure modes

The paper defines five failure modes based on joint movement of (Δσ_raw, ΔM*).
This is the monitoring system that tells operators *what kind* of problem they have.

In [ ]:
scenarios = [
    ("Healthy system",            0.00,  0.00),
    ("Agent improving",           0.05, -0.03),
    ("Agent degrading, masked",  -0.04,  0.05),
    ("Correlated drift",         -0.03, -0.02),
    ("Corrector coasting",        0.00,  0.04),
]

print(f"{'Scenario':<30} {'Δσ_raw':>8} {'ΔM*':>8} {'Diagnosis':<25} {'Action'}")
print("-" * 100)

actions = {
    FailureMode.HEALTHY: "No intervention needed",
    FailureMode.AGENT_IMPROVING: "Reduce corrector budget (save cost)",
    FailureMode.MASKING_DEGRADATION: "URGENT: retrain agent or add capacity",
    FailureMode.CORRELATED_DRIFT: "Input distribution shifted; retrain both",
    FailureMode.CORRECTOR_COASTING: "Check if agent is coasting on corrector",
}

for name, d_raw, d_m in scenarios:
    mode = diagnose_failure_mode(d_raw, d_m)
    print(f"{name:<30} {d_raw:>+8.2f} {d_m:>+8.2f} {mode.value:<25} {actions[mode]}")

## 7. Drift simulation: watching a pipeline degrade

Simulate the auto-responder drifting (model staleness, distribution shift)
while the quality checker stays stable. Classic masking scenario.

In [ ]:
# Pipeline with drift on the auto-responder
drift_pipeline = make_support_pipeline()
drift_pipeline.get_node("auto_responder").drift_rate = 0.008

config = SimulationConfig(n_steps=400, seed=42)
result = simulate_pipeline(drift_pipeline, config)

# Show snapshots
resp_idx = result.node_names.index("auto_responder")
qc_idx = result.node_names.index("quality_check")

print("Auto-responder with drift (μ=0.008):\n")
print(f"{'Step':>6} {'σ_raw(resp)':>12} {'σ_corr(resp)':>13} {'M*(resp)':>10} {'σ_raw(qc)':>12}")
print("-" * 56)
for t in [0, 50, 100, 150, 200, 250, 300, 350, 399]:
    sr = result.sigma_raw_history[t, resp_idx]
    sc = result.sigma_corr_history[t, resp_idx]
    m = result.masking_history[t, resp_idx]
    sr_qc = result.sigma_raw_history[t, qc_idx]
    print(f"{t:>6} {sr:>12.3f} {sc:>13.3f} {m:>10.2f} {sr_qc:>12.3f}")

print("\nσ_raw(resp) drops as the agent drifts.")
print("σ_corr(resp) drops slower — the corrector absorbs the degradation.")
print("M* rises — the masking gap widens.")
print("Downstream quality_check also degrades — cascade through the chain.")

## 8. Optimal stopping: when should the agent escalate?

Paper Demonstration 2: the follow-up agent should ask clarifying questions
until σ_raw(intent) exceeds a threshold, then stop.

Each question improves intent quality but risks user abandonment.

In [ ]:
# Optimal stopping model from the paper
sigma_skill_downstream = 0.55
D_downstream = 3  # layers after intent
p_abandon = 0.15  # per-question abandonment rate
delta_sigma = 0.12  # information gain per question

print("Optimal stopping for clarifying questions:\n")
print(f"{'Questions':>10} {'σ(intent)':>10} {'Survival':>10} {'Quality':>10} {'Net value':>10}")
print("-" * 55)

sigma_intent = 0.40  # ambiguous initial message
best_value = 0
best_n = 0

for n in range(6):
    sigma_n = min(sigma_intent + n * delta_sigma, 0.95)
    survival = (1 - p_abandon) ** n
    # Downstream quality: intent quality propagates through D layers
    quality = sigma_n * sigma_skill_downstream ** D_downstream
    value = quality * survival
    
    marker = " ← optimum" if value > best_value else ""
    if value > best_value:
        best_value = value
        best_n = n
    
    print(f"{n:>10} {sigma_n:>10.2f} {survival:>10.2f} {quality:>10.3f} {value:>10.3f}{marker}")

print(f"\nOptimal: ask {best_n} questions, then proceed.")
print(f"The {best_n+1}th question's information gain no longer offsets abandonment cost.")

## 9. Comparing escalation strategies

What if we change the escalation threshold? Too early = wasted human capacity.
Too late = poor customer experience from bad auto-responses.

In [ ]:
print("Effect of human review capacity at escalation node:\n")
print(f"{'Review K/N':>12} {'C_op':>8} {'B_eff':>8} {'T*_auto(esc)':>14} {'Feasible':>10}")
print("-" * 56)

for review_cap in [0.10, 0.25, 0.50, 0.75, 1.00]:
    p = make_support_pipeline()
    p.get_node("escalation").review_capacity = review_cap
    r = analyze_pipeline(p, p_min=0.75)
    
    esc_t = next(
        (s.t_auto for s in r.intervention_schedule if s.node_name == "escalation"),
        0.0
    )
    status = "YES" if r.feasibility.feasible else "NO"
    
    print(f"{review_cap:>12.2f} {r.feasibility.c_op:>8.3f} {r.feasibility.b_eff:>8.4f} {esc_t:>14.1f} {status:>10}")

print("\nMore human review at escalation raises C_op, but the bottleneck")
print("is upstream — improving the auto-responder has more leverage.")

## 10. Failure surface explanation

In [ ]:
# Re-run with p_min=0.75 to see the failure surface
p = make_support_pipeline()
r = analyze_pipeline(p, p_min=0.75)
print(r.failure_explanation)

## Key takeaways

1. **Chains are fragile** — masking compounds super-multiplicatively with depth. D_max ≈ 3-4 for typical parameters.
2. **Masking is the primary failure mode in support pipelines** — the quality checker hides auto-responder weakness. Dual-σ tracking catches it; monitoring only σ_corr does not.
3. **The diagnostic differential classifies failure modes** — (Δσ_raw, ΔM*) tells you whether the agent is degrading, the corrector is drifting, or both.
4. **Optimal stopping matters** — asking too many clarifying questions loses users; asking too few sends ambiguous intents downstream.
5. **Escalation capacity is not the bottleneck** — the upstream auto-responder is. Fix upstream, not downstream.